# Klasifikasi Status Perangkat dengan Model ML

*Referensi: Onno W. Purbo, IoT Terintegrasi dengan Machine Learning & Data Analytics, Bab 8.*

Notebook ini membandingkan Logistic Regression dan Random Forest untuk memprediksi tiga status perangkat: `normal`, `warning`, dan `fault`.

> **Pertanyaan ML:** model mana yang paling akurat dalam membedakan ketiga status pada test set?

## Tahapan Analisis

1. Menyiapkan fitur sensor dan label status.
2. Membagi data secara kronologis: 70% train, 15% validation, dan 15% test.
3. Melatih Logistic Regression dan Random Forest.
4. Membandingkan accuracy, confusion matrix, precision, recall, dan F1-score.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /Users/sulhanf/Second Brain/01 Projects/04 Katana/iot-ml-book


## 1. Menyiapkan Dataset Klasifikasi

In [2]:
import subprocess, sys
dataset_result = subprocess.run([sys.executable, "scripts/prepare_multiclass_status_dataset.py"], capture_output=True, text=True, check=True)
print(dataset_result.stdout)

Distribusi label:
status_label
normal     2048
warning      80
fault        30
Name: count, dtype: int64
File disimpan di: data/processed/classification_status



> ### Checkpoint 1 — Dataset Siap Digunakan
> Fitur, label, dan split time-aware telah dibuat. Distribusi label perlu diperhatikan karena kelas `warning` dan `fault` lebih sedikit daripada `normal`.

## 2. Logistic Regression

In [3]:
logreg_result = subprocess.run([sys.executable, "scripts/classification_logreg_status.py"], capture_output=True, text=True, check=True)
print(logreg_result.stdout)

=== LOGISTIC REGRESSION - VALIDATION ===
Accuracy: 0.9938271604938271

Confusion Matrix:
[[307   0   0]
 [  2   9   0]
 [  0   0   6]]

Classification Report:
              precision    recall  f1-score   support

       fault       1.00      1.00      1.00         6
      normal       0.99      1.00      1.00       307
     warning       1.00      0.82      0.90        11

    accuracy                           0.99       324
   macro avg       1.00      0.94      0.97       324
weighted avg       0.99      0.99      0.99       324


=== LOGISTIC REGRESSION - TEST ===
Accuracy: 0.9907407407407407

Confusion Matrix:
[[305   0   0]
 [  3  11   0]
 [  0   0   5]]

Classification Report:
              precision    recall  f1-score   support

       fault       1.00      1.00      1.00         5
      normal       0.99      1.00      1.00       305
     warning       1.00      0.79      0.88        14

    accuracy                           0.99       324
   macro avg       1.00      0.93 

## 3. Random Forest Classifier

Random Forest digunakan untuk menguji apakah hubungan non-linear dapat meningkatkan klasifikasi.

In [4]:
rf_result = subprocess.run([sys.executable, "scripts/classification_rf_status.py"], capture_output=True, text=True, check=True)
print(rf_result.stdout)

import re
from IPython.display import Markdown, display
logreg_test_accuracy = float(re.search(r"LOGISTIC REGRESSION - TEST ===\nAccuracy: ([0-9.]+)", logreg_result.stdout).group(1))
rf_test_accuracy = float(re.search(r"RANDOM FOREST CLASSIFIER - TEST ===\nAccuracy: ([0-9.]+)", rf_result.stdout).group(1))
best_name, best_accuracy = max(
    [("Logistic Regression", logreg_test_accuracy), ("Random Forest", rf_test_accuracy)],
    key=lambda item: item[1],
)
display(Markdown(
    f"> ### Checkpoint 2 — Model Terbaik pada Test Set\n"
    f"> **{best_name}** menghasilkan accuracy tertinggi, yaitu **{best_accuracy:.3f}**."
))

=== RANDOM FOREST CLASSIFIER - VALIDATION ===
Accuracy: 0.9907407407407407

Confusion Matrix:
[[306   1   0]
 [  2   9   0]
 [  0   0   6]]

Classification Report:
              precision    recall  f1-score   support

       fault       1.00      1.00      1.00         6
      normal       0.99      1.00      1.00       307
     warning       0.90      0.82      0.86        11

    accuracy                           0.99       324
   macro avg       0.96      0.94      0.95       324
weighted avg       0.99      0.99      0.99       324


=== RANDOM FOREST CLASSIFIER - TEST ===
Accuracy: 0.9845679012345679

Confusion Matrix:
[[304   1   0]
 [  4  10   0]
 [  0   0   5]]

Classification Report:
              precision    recall  f1-score   support

       fault       1.00      1.00      1.00         5
      normal       0.99      1.00      0.99       305
     warning       0.91      0.71      0.80        14

    accuracy                           0.98       324
   macro avg       0.97 

> ### Checkpoint 2 — Model Terbaik pada Test Set
> **Logistic Regression** menghasilkan accuracy tertinggi, yaitu **0.991**.

## Penutup

Kedua model menghasilkan accuracy tinggi pada validation dan test set. Confusion matrix menunjukkan bahwa kesalahan terutama terjadi pada kelas minoritas `warning` dan `fault`.

> **Kesimpulan:** accuracy saja belum cukup. Confusion matrix, recall, dan F1-score per kelas tetap diperlukan untuk menilai performa pada kelas minoritas.